## 7 - Baseline

Entraînement d'un modèle trivial servant de point de comparaison pour les modèles plus complexes.

- La baseline fixe un seuil minimal : tout modèle plus complexe doit la battre pour justifier son coût.
- Sans point de comparaison, un score MAE ou RMSE n'est pas interprétable en soi.
- `DummyRegressor` : stratégie à choisir entre moyenne et médiane selon la distribution de la cible.

La distribution de `SalePrice` est asymétrique à droite. Prédire la moyenne donnerait une MAE artificiellement élevée. La médiane est la constante mathématique qui minimise la MAE, c'est donc notre stratégie de baseline.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
from src.preprocessing import regrouper_categories_rares, traitement_valeurs_incohérentes, zero_vers_nan, Transformation_binaire, remplacer_au_dessus_seuil, fusionner_categories, extraire_dates

# 1. Chargement
df = pd.read_csv("../data/raw/bluebook-for-bulldozers/TrainAndValid.csv", low_memory=False)
df["saledate"] = pd.to_datetime(df["saledate"])

# 2. Nettoyage global
taux_nan = df.isnull().mean()
df = df.drop(columns=taux_nan[taux_nan > 0.7].index)
df = traitement_valeurs_incohérentes(df, "YearMade")
df = zero_vers_nan(df, "MachineHoursCurrentMeter")
df = Transformation_binaire(df, "MachineHoursCurrentMeter", "hours_reported")
df = remplacer_au_dessus_seuil(df, "MachineHoursCurrentMeter", 40000)

# 3. Split temporel
train = df[df["saledate"] < "2012-01-01"].copy()
test = df[df["saledate"] >= "2012-01-01"].copy()

# 4. Transformations sur train/test séparés (pour éviter la fuite de données)
train, test = regrouper_categories_rares(train, test, "Hydraulics", 500)
train, test = regrouper_categories_rares(train, test, "fiProductClassDesc", 500)

mapping_enclosure = {"NO ROPS": "OROPS", "EROPS AC": "EROPS w AC", "None or Unspecified": np.nan}
train = fusionner_categories(train, "Enclosure", mapping_enclosure)
test = fusionner_categories(test, "Enclosure", mapping_enclosure)

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# 1. Séparation X et y (en excluant SalePrice et MachineID)
X_train = train.drop(columns=["SalePrice", "MachineID"])
y_train = train["SalePrice"]

X_test = test.drop(columns=["SalePrice", "MachineID"])
y_test = test["SalePrice"]

# 2. Instanciation et entraînement de la baseline
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_train, y_train)

# 3. Prédictions
y_pred_train_dummy = dummy_median.predict(X_train)
y_pred_test_dummy = dummy_median.predict(X_test)

# 4. Évaluation (MAE)
mae_train_dummy = mean_absolute_error(y_train, y_pred_train_dummy)
mae_test_dummy = mean_absolute_error(y_test, y_pred_test_dummy)

print(f"Baseline MAE Train : {mae_train_dummy:.2f}")
print(f"Baseline MAE Test  : {mae_test_dummy:.2f}")

Baseline MAE Train : 16454.19
Baseline MAE Test  : 19697.33


Saledate est au format date hors les modèles que l'on souhaite utiliser n'accepte que des valeurs numériques. On va extraire l'année, le mois ,le trimestre 

In [3]:
X_train = extraire_dates(X_train, "saledate" )
X_test = extraire_dates(X_test, "saledate" )


On va encoder les variables non numérique afin de pouvoir tester nos différents modèles. Avant cela nous allons effectuer un dernier tri dans celles-ci. 

On supprime les variables redondantes ou encore celle qui sont utilisée comme des identifiants 


In [4]:
colonnes_a_supprimer = [
    'SalesID', 'ModelID', 'datasource', 'auctioneerID', 
    'fiModelDesc', 'fiBaseModel', 'fiSecondaryDesc', 
    'ProductGroupDesc', 'ProductSize'
]

X_train = X_train.drop(columns=colonnes_a_supprimer)
X_test = X_test.drop(columns=colonnes_a_supprimer)



In [5]:
print(X_train.shape)

(401125, 15)


In [6]:
colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()
print(colonnes_texte)
print(f"Nombre de colonnes à encoder : {len(colonnes_texte)}")

['fiProductClassDesc', 'state', 'ProductGroup', 'Enclosure', 'Forks', 'Ride_Control', 'Transmission', 'Hydraulics', 'Coupler']
Nombre de colonnes à encoder : 9


C:\Users\Abram\AppData\Local\Temp\ipykernel_23952\1769016324.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()


In [7]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [8]:
X_train.columns.tolist()

['YearMade',
 'MachineHoursCurrentMeter',
 'fiProductClassDesc',
 'state',
 'ProductGroup',
 'Enclosure',
 'Forks',
 'Ride_Control',
 'Transmission',
 'Hydraulics',
 'Coupler',
 'hours_reported',
 'saleyear',
 'salemonth',
 'salequarter']

In [9]:
y_train.shape


(401125,)

In [10]:
y_test.shape

(11573,)